# V28: unchanged native batches with CPU prefetch

One bounded CPU producer prepares the next batch while the consumer runs the
original model. Packing remains8192 and individual conversation cap4096.
CPU proofs measure batch identity, not GPU equivalence, speed or score.
[Plan](../docs/SPAN_028_PLAN.md) · [CPU proof](../docs/SPAN_028_CPU_PROOF.md) ·
[Preserved packing failure](../docs/SPAN_027_RESULTS.md).

In [1]:
from pathlib import Path
import json
import pandas as pd
REPO=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PUB=REPO/'results/span_028';proof=json.loads((PUB/'cpu_audit.json').read_text())
rows=[]
for role,cohorts in proof['actual_native_cohorts'].items():
 for name,v in cohorts.items():
  rows.append({'role':role,'cohort':name,'rows':v['rows'],'microbatches':v['micro_batches'],
   'host_batches_exact':v['exact_host_batches'],'row_order_exact':v['exact_row_order'],
   'positions_exact':v['exact_positions'],'local_sync_preparation_s':v['local_synchronous_preparation_seconds']})
display(pd.DataFrame(rows))
print('CPU proof elapsed:',proof['elapsed_cpu_seconds'],'seconds; cap:',proof['cpu_cap_seconds'])
print('Packing:',proof['packing_tokens'],'; producer threads:',proof['producer_threads'],'; queue:',proof['queue_depth'])
print('No labels, GPU inference, probability identity, speedup or quality measured.')
print('Preparation times are laptop measurements during an audit, not expected Kaggle savings.')
for name in ['prefetch_tests.json','profile_tests.json']:
 path=PUB/name
 if path.exists():print(name,json.loads(path.read_text()))
decision=PUB/'profile_audit.json'
if decision.exists():
 result=json.loads(decision.read_text());print('Engineering decision:',result)
else:print('GPU source/cell rehearsals, admission, inference and engineering audit still pending.')
print('H73 fixed60/40prior remains deferred until the engineering prerequisite passes.')

,role,cohort,rows,microbatches,host_batches_exact,row_order_exact,positions_exact,local_sync_preparation_s
0,m0,warmup,80,9,True,True,True,0.447334
1,m0,control,256,33,True,True,True,1.088361
2,m0,uniform,512,67,True,True,True,2.468255
3,m0,long,64,63,True,True,True,2.043294
4,m0,test,3,1,True,True,True,0.022970
5,m3,warmup,80,9,True,True,True,0.304466
6,m3,control,256,33,True,True,True,1.082574
7,m3,uniform,512,65,True,True,True,2.290881
8,m3,long,64,63,True,True,True,1.953235
9,m3,test,3,1,True,True,True,0.029615


CPU proof elapsed: 31.847062963061035 seconds; cap: 300
Packing: 8192 ; producer threads: 1 ; queue: 2
No labels, GPU inference, probability identity, speedup or quality measured.
Preparation times are laptop measurements during an audit, not expected Kaggle savings.
prefetch_tests.json {'passed': True, 'tests': 7, 'runner': 'PYTHONPATH=src .venv/bin/python -m unittest discover -s tests -p test_cpu_prefetch.py -v', 'observed_test_seconds': 0.104, 'source_sha256': {'src/arena/cpu_prefetch.py': 'da7502af819a86c421717f68d40c4a9892cad60bc2f49279247805697be30c35', 'tests/test_cpu_prefetch.py': '54fab79e57894f8b0761518ab65e48335c7622cf8e9b55e2fc630774b6cbdcc5'}}
profile_tests.json {'passed': True, 'tests': 6, 'runner': 'PYTHONPATH=src .venv/bin/python -m unittest discover -s tests -p test_span028_profile.py -v', 'observed_test_seconds': 0.133, 'source_sha256': {'src/arena/span028_checks.py': '3a685a55eb37d053caa7305ab8a56f07117b43e345b69933fef77fb70e680c48', 'src/arena/span028_profile.py': '